<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Algopro/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datetime import date, timedelta
import random
import math

In [2]:
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
chars = "0123456789-|>,. "
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
VOCAB_SIZE = len(chars)
PAD_ID = stoi[" "]

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print(f"Vocab size: {VOCAB_SIZE}")

Vocab size: 16


In [4]:
START_DATE = date(2020, 1, 1)
END_DATE = date(2024, 12, 31)
DATE_RANGE_DAYS = (END_DATE - START_DATE).days

COMPONENT_WIDTH = 4

# format: "YYYY-MM-DD|YYYY-MM-DD>AAAA,BBBB,CCCC."
# where AAAA = days-from-START_DATE for d1 (reversed)
#       BBBB = days-from-START_DATE for d2 (reversed)
#       CCCC = abs difference (reversed)
INPUT_PREFIX_LEN = 22  # "YYYY-MM-DD|YYYY-MM-DD>"
ANSWER_LEN = 3 * COMPONENT_WIDTH + 2 + 1  # 3 numbers + 2 commas + 1 period
SEQ_LEN = INPUT_PREFIX_LEN + ANSWER_LEN

def random_date():
    offset = random.randint(0, DATE_RANGE_DAYS)
    return START_DATE + timedelta(days=offset)

def fmt(n):
    return f"{n:0{COMPONENT_WIDTH}d}"[::-1]

def parse_component(s):
    return int(s[::-1])

def make_example():
    d1, d2 = random_date(), random_date()
    if d1 > d2:
        d1, d2 = d2, d1
    o1 = (d1 - START_DATE).days
    o2 = (d2 - START_DATE).days
    diff = o2 - o1
    s = f"{d1.isoformat()}|{d2.isoformat()}>{fmt(o1)},{fmt(o2)},{fmt(diff)}."
    assert len(s) == SEQ_LEN, f"Expected {SEQ_LEN}, got {len(s)}: {s!r}"
    return s

for _ in range(3):
    print(make_example())
print(f"\nSeq len: {SEQ_LEN}, answer portion: {ANSWER_LEN}")

2022-02-27|2024-09-25>8870,9271,1490.
2024-04-01|2024-12-28>2551,3281,1720.
2020-03-23|2022-05-11>2800,1680,9770.

Seq len: 37, answer portion: 15


In [6]:
def make_batch(batch_size):
    examples = [make_example() for _ in range(batch_size)]
    encoded = torch.tensor([encode(s) for s in examples], dtype=torch.long)

    # input: all but last token
    x = encoded[:, :-1]
    # target: all but first token
    y = encoded[:, 1:].clone()

    # mask out loss on the input prefix, we only care about predicting the answer tokens
    loss_mask = torch.zeros_like(y, dtype=torch.bool)
    loss_mask[:, INPUT_PREFIX_LEN - 1:] = True
    return x.to(device), y.to(device), loss_mask.to(device)

x, y, m = make_batch(2)
print(f"x shape: {x.shape}, y shape: {y.shape}")
print(f"x[0]: {decode(x[0].tolist())!r}")
print(f"y[0]: {decode(y[0].tolist())!r}")

x shape: torch.Size([2, 36]), y shape: torch.Size([2, 36])
x[0]: '2024-05-24|2024-08-26>5061,9961,4900'
y[0]: '024-05-24|2024-08-26>5061,9961,4900.'


In [42]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        # projections (no bias needed)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape  # batch, time, channels (d_model)

        # project and reshape to (B, n_heads, T, d_head)
        Q = self.W_q(x).reshape(B, T, self.d_head, self.n_heads).permute(0, 3, 1, 2)
        K = self.W_k(x).reshape(B, T, self.d_head, self.n_heads).permute(0, 3, 1, 2)
        V = self.W_v(x).reshape(B, T, self.d_head, self.n_heads).permute(0, 3, 1, 2)

        # attention scores: (B, n_heads, T, T)
        scores = Q @ K.transpose(-1, -2)

        # causal mask: position i can only attend to positions <= i
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool().to(device)
        scores = torch.masked_fill(scores, mask, float('-inf')) / (self.d_model ** 0.5)
        
        # (B, n_heads, T, T)
        attn = torch.softmax(scores, dim=-1)
        # (B, n_heads, T, d_head)
        out = attn @ V

        # concat heads back to (B, T, d_model)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return out

In [43]:
# changing token i should NOT change the output at any position < i.

mha = MultiHeadSelfAttention(d_model=64, n_heads=4).to(device)
mha.eval()
with torch.no_grad():
    x_a = torch.randn(1, 5, 64).to(device)
    x_b = x_a.clone()
    x_b[0, 3:] = torch.randn(2, 64)  # change tokens 3 and 4

    out_a = mha(x_a)
    out_b = mha(x_b)

    # outputs at positions 0, 1, 2 should be identical
    diff_early = (out_a[0, :3] - out_b[0, :3]).abs().max().item()
    diff_late = (out_a[0, 3:] - out_b[0, 3:]).abs().max().item()
    print(f"Max diff at positions 0-2 (should be = 0): {diff_early:.2}")
    print(f"Max diff at positions 3-4 (should be > 0): {diff_late:.2}")

Max diff at positions 0-2 (should be = 0): 0.0
Max diff at positions 3-4 (should be > 0): 0.48


In [46]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        ) 
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # pre-norm variant: LayerNorm before sublayer, residual around it (more stable for training than the original post-norm)
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

block = TransformerBlock(d_model=64, n_heads=4, d_ff=256).to(device)
print(f"Block output shape: {block(torch.randn(2, 10, 64).to(device)).shape}")

Block output shape: torch.Size([2, 10, 64])


In [94]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.max_len = max_len

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=device, dtype=torch.long).unsqueeze(0) # (1, T)

        # initial h
        h = self.tok_emb(x) + self.pos_emb(pos)

        # go through transformer block
        for block in self.blocks:
            h = block(h)

        # normalize
        h = self.head(self.ln_f(h))

        return h

model = TinyTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=32,
    n_heads=16,
    n_layers=1,
    d_ff=256,
    max_len=SEQ_LEN,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

Model parameters: 22,160


In [95]:
x, y, loss_mask = make_batch(256)
x, y = x.to(device), y.to(device)
model(x).shape, y.shape, loss_mask.shape

(torch.Size([256, 36, 16]), torch.Size([256, 36]), torch.Size([256, 36]))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

BATCH_SIZE = 256
N_STEPS = 9000
EVAL_EVERY = 1000

model.train()
for step in range(N_STEPS):
    x, y, loss_mask = make_batch(BATCH_SIZE)
    x, y = x.to(device), y.to(device)
    logits = model(x)
    logits = logits.view(-1, logits.size(-1))  
    y = y.view(-1)                  
    loss_mask = loss_mask.view(-1).bool()                

    # loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), reduction='none')
    # loss = (loss * loss_mask.view(-1)).sum() / loss_mask.sum()

    loss = F.cross_entropy(
        logits[loss_mask],
        y[loss_mask]
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % EVAL_EVERY == 0 or step == N_STEPS - 1:
        print(f"step {step:5d} | loss {loss.item():.4f}")

step     0 | loss 0.1545
step  1000 | loss 0.0909
step  2000 | loss 0.0007
step  2999 | loss 0.0050


In [99]:
@torch.no_grad()
def predict(model, d1: date, d2: date, return_full=False):
    model.eval()
    if d1 > d2:
        d1, d2 = d2, d1
    prompt = f"{d1.isoformat()}|{d2.isoformat()}>"
    ids = encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(ANSWER_LEN):
        logits = model(x)
        next_id = logits[0, -1].argmax().item()
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)
    answer_str = decode(x[0, INPUT_PREFIX_LEN:].tolist())
    if return_full:
        return answer_str

    try:
        parts = answer_str.rstrip(".").split(",")
        if len(parts) != 3:
            return None
        return parse_component(parts[2])
    except (ValueError, IndexError):
        return None

print("Sample outputs:")
for _ in range(3):
    d1, d2 = random_date(), random_date()
    full = predict(model, d1, d2, return_full=True)
    true_diff = abs((d2 - d1).days)
    print(f"  {d1} {d2} -> {full!r} (true diff: {true_diff})")

N_EVAL = 500
correct = 0
for _ in range(N_EVAL):
    d1, d2 = random_date(), random_date()
    true_diff = abs((d2 - d1).days)
    pred = predict(model, d1, d2)
    if pred == true_diff:
        correct += 1

print(f"\nAccuracy: {correct}/{N_EVAL} = {100*correct/N_EVAL:.1f}%")

Sample outputs:
  2021-07-01 2021-12-16 -> '7450,5170,8610.' (true diff: 168)
  2024-12-29 2022-07-09 -> '0290,4281,4090.' (true diff: 904)
  2023-04-02 2024-10-06 -> '7811,0471,3550.' (true diff: 553)

Accuracy: 472/500 = 94.4%


In [100]:
test_pairs = [
    (date(2020, 1, 1), date(2020, 12, 31)),  # 365 (leap year)
    (date(2021, 1, 1), date(2021, 12, 31)),  # 364
    (date(2020, 2, 28), date(2020, 3, 1)),   # 2 (leap)
    (date(2021, 2, 28), date(2021, 3, 1)),   # 1 (non-leap)
    (date(2022, 6, 15), date(2024, 6, 15)),  # 731
    (date(2024, 3, 25), date(2024, 11, 13)), # 233
]

for d1, d2 in test_pairs:
    true_diff = abs((d2 - d1).days)
    pred = predict(model, d1, d2)
    mark = "✓" if pred == true_diff else "✗"
    print(f"{mark} {d1} to {d2}: predicted {pred}, true {true_diff}")

✗ 2020-01-01 to 2020-12-31: predicted 265, true 365
✓ 2021-01-01 to 2021-12-31: predicted 364, true 364
✓ 2020-02-28 to 2020-03-01: predicted 2, true 2
✓ 2021-02-28 to 2021-03-01: predicted 1, true 1
✗ 2022-06-15 to 2024-06-15: predicted 831, true 731
✓ 2024-03-25 to 2024-11-13: predicted 233, true 233
